# SIPTA -- Ingesta y EDA: Finanzas, Economía Informal (RIVI) e Inversión
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona C (Sofía Hidalgo -- Ingesta & EDA)**  
**Colaboración / Integración**: Persona A (Adan Sánchez -- Lead Data Engineer)  
**Objetivo**: Consolidación de series semestrales RIVI, puntos de encuentro IPES, inversión educativa SED y análisis exploratorio (EDA).  
**Datos de Entrada**: `data/raw/FINANZAS_INVERSION_PUBLICA/*`  
**Datos de Salida**: `data/processed/FINANZAS_INVERSION_PUBLICA/*`


## 1. Ingesta y Consolidación de Series RIVI y Puntos de Encuentro



# SIPTA Notebook: Ingestión de datos -- Dominio Finanzas / Economía Informal (D6)
Este notebook documenta la lectura, consolidación temporal de vendedores informales y la carga de infraestructura productiva (Puntos de Encuentro IPES).

### Objetivos
- Consolidar las 6 series semestrales de **Vendedores Informales (RIVI)** de 2017 a 2019.
- Cargar y estructurar los **Puntos de Encuentro de Vendedores** desde Excel/GeoJSON.
- Validar esquemas, llaves territoriales de localidad y anomalías en coordenadas.
- Exportar las copias versionadas a `data/raw/FINANZAS/`.

In [ ]:
import sys
from pathlib import Path
import re
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        break
RAW_DIR = ROOT / 'data' / 'raw'
FINANZAS_DIR = RAW_DIR / 'FINANZAS_INVERSION_PUBLICA' if (RAW_DIR / 'FINANZAS_INVERSION_PUBLICA').exists() else RAW_DIR / 'FINANZAS'
FINANZAS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio Raw Finanzas: {FINANZAS_DIR}")



## 1. Consolidación de Series: Vendedores Informales (RIVI)

In [ ]:
def consolidate_vendedores_informales(input_dir: Path) -> pd.DataFrame:
    """Lee los archivos txt de RIVI, extrae la fecha del nombre y estandariza columnas."""
    files = sorted(list(input_dir.glob("rivi-numero-vendedores-informales-localidad-*.txt")))
    if not files:
        files = sorted(list((RAW_DIR / 'FINANZAS_INVERSION_PUBLICA').glob("rivi-numero-vendedores-informales-localidad-*.txt")))
    if not files:
        files = sorted(list(RAW_DIR.rglob("rivi-numero-vendedores-informales-localidad-*.txt")))
    
    assert len(files) > 0, f"No se encontraron archivos RIVI en {input_dir}"
    
    dfs = []
    for file in files:
        match = re.search(r'(\d{4}-\d{2}-\d{2})', file.name)
        fecha = match.group(1) if match else "Desconocida"
        
        try:
            df = pd.read_csv(file, sep='\t', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file, sep='\t', encoding='latin1')
            
        if 'IndiceRespuesta' in df.columns:
            df = df.rename(columns={
                'IndiceRespuesta': 'codigo_localidad',
                'NumeroLocalidad': 'nombre_localidad',
                'Numero': 'numero_vendedores',
                'Porcentaje': 'porcentaje'
            })
        elif 'NombreLocalidad' in df.columns:
            df = df.rename(columns={
                'NumeroLocalidad': 'codigo_localidad',
                'NombreLocalidad': 'nombre_localidad',
                'Numero': 'numero_vendedores',
                'Porcentaje': 'porcentaje'
            })
            
        df_clean = df[['codigo_localidad', 'nombre_localidad', 'numero_vendedores', 'porcentaje']].copy()
        df_clean['fecha_corte'] = fecha
        df_clean['archivo_origen'] = file.name
        dfs.append(df_clean)
        
    df_final = pd.concat(dfs, ignore_index=True)
    return df_final

df_vendedores = consolidate_vendedores_informales(FINANZAS_DIR)
print(f"Vendedores Informales consolidados: {len(df_vendedores)} filas.")
display(df_vendedores.head())

## 2. Ingesta: Puntos de Encuentro de Vendedores (IPES)

In [ ]:
excel_path = FINANZAS_DIR / 'Punto de encuentro vendedores. Bogotá D.C..xlsx'
if not excel_path.exists():
    excel_path = RAW_DIR / 'Punto de encuentro vendedores. Bogotá D.C..xlsx'

df_puntos = pd.read_excel(excel_path)
print(f"[OK] Puntos de encuentro cargados: {len(df_puntos)} registros.")

cols_puntos = [
    'properties/CPUNNOM',
    'properties/CPUNDIR',
    'properties/CPUNLOC',
    'properties/CPUNBARRIO',
    'properties/CPUNHORAPV',
    'properties/CPUNNUMLOC'
]
display(df_puntos[cols_puntos].head())


In [ ]:
print("=== REVISIÓN DE LLAVES Y COBERTURA ===")
print("Fechas RIVI:", sorted(df_vendedores['fecha_corte'].unique()))
print("Localidades en Puntos de Encuentro:", df_puntos['properties/CPUNLOC'].unique().tolist())

# Guardar copias consolidadas en raw
df_vendedores.to_csv(FINANZAS_DIR / "vendedores_informales_consolidado.csv", index=False, encoding='utf-8')
df_puntos.to_csv(FINANZAS_DIR / "puntos_encuentro_vendedores.csv", index=False, encoding='utf-8')
print("[OK] Respaldos generados en data/raw/FINANZAS/")


### Notas de Ingesta -- Finanzas / Economía (D6)
1. **Vendedores Informales (RIVI):**
   - 126 registros semestrales (2017-06-30 a 2019-12-30).
   - Llave territorial: `codigo_localidad` (1 a 20 y código 160).
2. **Puntos de Encuentro (IPES):**
   - 4 equipamientos comerciales regulados de apoyo al vendedor informal (Alcalá, Aguas, Mundo Aventura y Tintal).
   - **⚠ Pendiente de validar con datos (Inconsistencia en código de localidad):** La columna `CPUNNUMLOC` registra '18' para Alcalá (Usaquén) y '12' para Aguas (Santa Fe); se corregirá mapeando con `CPUNLOC` en la fase 1.4 de limpieza.
   - **⚠ Pendiente de validar con datos (Escala decimal de coordenadas):** Las columnas `geometry/coordinates` en Excel perdieron el punto decimal (`-7405083...`); se normalizarán a coordenadas geográficas WGS84 en la limpieza.

## 2. Análisis Exploratorio de Economía Informal e Inversión



# 06 · EDA Finanzas e inversión pública

Una fuente en `data/raw/FINANZAS_INVERSION_PUBLICA/`:
- `inversion_educacion_por_localidad_12_2025.gpkg` -- territorialización de la inversión en
  educación por localidad (capa `sed_2026__territorializacioninversion2025`, 20 polígonos,
  EPSG:3857), con asignado, ejecutado y girado.

**Indicadores objetivo**: FIN-01 (ejecución presupuestal por localidad), FIN-02 (girado vs
asignado). Las unidades parecen ser pesos COP; validar con la SED.

## 0. Configuración

In [ ]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

# Búsqueda universal de la raíz del proyecto (Local, Subdirectorios y Google Colab)
ROOT = None
for _p in [Path(".").resolve(), Path("..").resolve(), Path("../..").resolve(), Path("/content/DataJam_DataOlinguitos_Gen"), Path("/content")]:
    if (_p / "src" / "eda").exists() or (_p / "src").exists():
        ROOT = _p
        break

if ROOT is None:
    ROOT = Path(".").resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")


## 0.1 Fuentes del sector en el catálogo

In [ ]:
cat = eda.load_catalog()
sec = cat[cat["indicadores"].astype(str).str.contains("FIN", case=False, na=False)]
display(sec[["id", "nombre", "archivo", "temporalidad", "indicadores", "valor_publico"]])


## 1. Inversión en educación por localidad

### Territorialización de inversión en educación (2025)

In [ ]:
t0('inversion')
SPEC = {
    'id': 'inversion_educacion_2025',
    'titulo': 'Territorialización de inversión en educación (2025)',
    'path': 'data/raw/FINANZAS_INVERSION_PUBLICA/inversion_educacion_por_localidad_12_2025.gpkg',
    'capa': 'sed_2026__territorializacioninversion2025',
    'origen': 'Secretaría de Educación del Distrito (SED)',
    'corte': 'Dic 2025 (nombre del archivo)',
    'valor_publico': 'Asignación y ejecución de inversión en educación por localidad',
    'indicadores': 'FIN-01 (ejecución presupuestal), FIN-02 (girado)',
    'notas': '20 polígonos (MultiPolygon, EPSG:3857). Columnas: R_ASIGNADOS, R_EJECUTADOS, R_GIRADOS, COD_LOCA. COD_LOCA numérico 1-20; unidades por validar (pesos COP).',
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('inversion')


## 2. Análisis de ejecución presupuestal

### 2.1 Asignado, ejecutado y girado por localidad

In [ ]:
from src.eda.profiling import localidad_de_codigo
fin = gpd.read_file(str(RAW_DIR / "FINANZAS_INVERSION_PUBLICA" / "inversion_educacion_por_localidad_12_2025.gpkg"), layer="sed_2026__territorializacioninversion2025")
fin["LOCALIDAD"] = fin["COD_LOCA"].map(localidad_de_codigo)
fin["PCT_EJEC"] = fin["R_EJECUTADOS"] / fin["R_ASIGNADOS"]
fin["PCT_GIRADO"] = fin["R_GIRADOS"] / fin["R_ASIGNADOS"]
tabla = fin[["LOCALIDAD", "R_ASIGNADOS", "R_EJECUTADOS", "R_GIRADOS", "PCT_EJEC", "PCT_GIRADO"]].sort_values("R_ASIGNADOS", ascending=False)
tabla.to_csv(REPORTS / "finanzas_inversion_por_localidad.csv", index=False)
display(tabla)
print(f"Total asignado: {fin['R_ASIGNADOS'].sum():,.0f} | ejecutado: {fin['R_EJECUTADOS'].sum():,.0f} ({fin['R_EJECUTADOS'].sum() / fin['R_ASIGNADOS'].sum():.1%})")
print(f"Ejecución promedio por localidad: {fin['PCT_EJEC'].mean():.1%} (min {fin['PCT_EJEC'].min():.1%}, max {fin['PCT_EJEC'].max():.1%})")
tabla.plot(x="LOCALIDAD", y="PCT_EJEC", kind="bar", figsize=(11, 4), title="% de ejecución por localidad", color="#4c72b0")
plt.ylabel("% ejecutado")
plt.show()


### 2.2 Mapa temático de ejecución

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
fin.plot(ax=ax, column="PCT_EJEC", cmap="RdYlGn", legend=True, edgecolor="white", linewidth=0.4, legend_kwds={"label": "% ejecutado", "shrink": 0.6})
ax.set_title("Ejecución de inversión en educación por localidad (2025)")
ax.set_axis_off()
plt.show()


## 3. Indicadores del sector

In [ ]:
sts = eda.indicator_status()
sts_sec = sts[sts["indicador"].str.startswith("FIN", na=False)]
display(sts_sec[["indicador", "dimension", "estado", "que_falta"]])
sts_sec.to_csv(REPORTS / "indicadores_finanzas.csv", index=False)
guardar_tiempos("06_eda_finanzas.csv")
print("Secciones del notebook:", list(_SECTION_T.keys()))
